In [1]:
import platform
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno 

warnings.filterwarnings("ignore")

# 재현성: 같은 난수를 항상 같게 만들어 결과가 매번 동일하도록 고정합니다.
np.random.seed(42)

# 한글 폰트 설정 (운영체제별 분기)
system = platform.system()
if system == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
else:
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)

준비 완료! 라이브러리 버전을 확인합니다.
numpy : 2.4.6
pandas: 3.0.3


In [2]:
def missing_summary(df):
    s = df.isnull().sum()
    p = (df.isnull().mean() * 100).round(2)
    out = pd.DataFrame({"missing": s, "missing_pct(%)": p})
    return out[out["missing"] > 0].sort_values("missing", ascending=False)


In [3]:
def detect_outliers_iqr(series, k=1.5):
    '''IQR 방법으로 이상치 마스크와 경계를 반환합니다.

    Parameters
    ----------
    series : pd.Series  — 수치형 시리즈
    k      : float      — 경계 폭의 IQR 배수 (기본 1.5, 더 엄격하게 보려면 3)

    Returns
    -------
    mask   : pd.Series(bool)  — 이상치 위치 (True)
    bounds : (lower, upper)   — 경계값
    '''
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    mask = (series < lower) | (series > upper)
    return mask, (lower, upper)

In [4]:
# 새 데이터셋 — '옷장패션' 주문 (가상)
np.random.seed(11)
n = 1500

partner = pd.DataFrame({
    "order_id": [f"K{str(i).zfill(5)}" for i in range(1, n + 1)],
    "customer_age": np.random.normal(33, 8, n).round().astype(int),
    "category": np.random.choice(["상의", "하의", "신발", "액세서리"], n, p=[0.35, 0.3, 0.2, 0.15]),
    "channel": np.random.choice(["web", "app"], n, p=[0.4, 0.6]),
    "price": np.random.choice([15900, 29900, 49900, 79900, 129900], n),
    "quantity": np.random.choice([1, 1, 1, 2, 2, 3], n),
})
partner["amount"] = partner["price"] * partner["quantity"]
partner["return_amount"] = np.where(
    np.random.rand(n) < 0.07, partner["amount"] * np.random.uniform(0.5, 1.0, n), 0
).round(0)

# 오염 심기
# (a) 나이 이상치 — 입력 실수(0, 999)
partner.loc[partner.sample(3, random_state=1).index, "customer_age"] = 999
partner.loc[partner.sample(2, random_state=2).index, "customer_age"] = 0

# (b) amount 결측 — app 채널에 더 자주 (MAR 시그널)
app = partner["channel"] == "app"
partner.loc[partner[app].sample(frac=0.05, random_state=3).index, "amount"] = np.nan
partner.loc[partner[~app].sample(frac=0.01, random_state=4).index, "amount"] = np.nan

# (c) return_amount 결측은 그대로 (0=환불없음)이라 결측 아님. 단, '관찰 안 됨'을 의도적으로 표현하기 위해
#     price 결측 5건 추가(접속 시점 가격이 누락된 사례)
partner.loc[partner.sample(5, random_state=5).index, "price"] = np.nan

# (d) quantity 이상치(단일 소비자 200개)
partner.loc[partner.sample(1, random_state=6).index, "quantity"] = 200

# (e) amount 극단값(50,000,000짜리 한 건 — '도매 의심')
partner.loc[partner.sample(1, random_state=7).index, "amount"] = 50_000_000

print("옷장패션 데이터 준비 완료:", partner.shape)
partner.head()

옷장패션 데이터 준비 완료: (1500, 8)


,order_id,customer_age,category,channel,price,quantity,amount,return_amount
0,K00001,47,신발,app,29900.0,2,59800.0,45445.0
1,K00002,31,상의,app,129900.0,3,389700.0,0.0
2,K00003,29,상의,web,49900.0,2,99800.0,0.0
3,K00004,12,상의,web,49900.0,3,149700.0,0.0
4,K00005,33,하의,app,129900.0,1,129900.0,0.0


In [5]:
# 시나리오 1 — 진단
print("shape:", partner.shape)
partner.info()
display(partner.describe())

shape: (1500, 8)
<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       1500 non-null   str    
 1   customer_age   1500 non-null   int64  
 2   category       1500 non-null   str    
 3   channel        1500 non-null   str    
 4   price          1495 non-null   float64
 5   quantity       1500 non-null   int64  
 6   amount         1449 non-null   float64
 7   return_amount  1500 non-null   float64
dtypes: float64(3), int64(2), str(3)
memory usage: 93.9 KB


,customer_age,price,quantity,amount,return_amount
count,1500.000000,1495.000000,1500.000000,1.449000e+03,1500.000000
mean,34.903333,60960.869565,1.789333,1.350475e+05,6381.010000
std,43.936525,40275.103681,5.173406,1.313695e+06,28315.037316
min,0.000000,15900.000000,1.000000,1.590000e+04,0.000000
25%,28.000000,29900.000000,1.000000,4.770000e+04,0.000000
50%,33.000000,49900.000000,2.000000,7.990000e+04,0.000000
75%,38.250000,79900.000000,2.000000,1.299000e+05,0.000000
max,999.000000,129900.000000,200.000000,5.000000e+07,322778.000000


In [6]:
# 결측 진단
print("[열별 결측]")
print(missing_summary(partner))

# 결측이 채널과 관련 있는지 (MAR 신호 검사)
amt_null = partner[partner["amount"].isnull()]
print("\n[amount 결측 행의 채널 분포]")
print(amt_null["channel"].value_counts(normalize=True).round(2))
print("\n[전체 채널 분포]")
print(partner["channel"].value_counts(normalize=True).round(2))

[열별 결측]
        missing  missing_pct(%)
amount       51            3.40
price         5            0.33

[amount 결측 행의 채널 분포]
channel
app    0.88
web    0.12
Name: proportion, dtype: float64

[전체 채널 분포]
channel
app    0.61
web    0.39
Name: proportion, dtype: float64


In [7]:
# IQR 이상치 — 수치형 컬럼 일괄 점검
num_cols = ["customer_age", "price", "quantity", "amount", "return_amount"]
print("[IQR 기준 이상치 개수]")
for c in num_cols:
    mask, (lo, up) = detect_outliers_iqr(partner[c].dropna())
    print(f"  {c:15s}  하한={lo:>12.1f}  상한={up:>12.1f}  이상치={mask.sum()}건")



[IQR 기준 이상치 개수]
  customer_age     하한=        12.6  상한=        53.6  이상치=18건
  price            하한=    -45100.0  상한=    154900.0  이상치=0건
  quantity         하한=        -0.5  상한=         3.5  이상치=1건
  amount           하한=    -75600.0  상한=    253200.0  이상치=145건
  return_amount    하한=         0.0  상한=         0.0  이상치=122건


In [8]:
Q1 = partner["amount"].quantile(0.25) #amount 이상치 실제 값 확인
Q3 = partner["amount"].quantile(0.75)
IQR = Q3 - Q1
lo = Q1 - 1.5 * IQR
up = Q3 + 1.5 * IQR

outliers = partner[(partner["amount"] < lo) | (partner["amount"] > up)]
outliers[["amount"]].sort_values("amount", ascending=False).head(10)

,amount
547,50000000.0
6,389700.0
12,389700.0
17,389700.0
26,389700.0
45,389700.0
34,389700.0
98,389700.0
467,389700.0
487,389700.0


In [9]:
(partner["amount"] == 389700.0).sum() #동일가격 이상치확인
partner[partner["amount"] == 389700.0].head(10)

,order_id,customer_age,category,channel,price,quantity,amount,return_amount
1,K00002,31,상의,app,129900.0,3,389700.0,0.0
6,K00007,29,액세서리,web,129900.0,3,389700.0,0.0
12,K00013,39,신발,app,129900.0,3,389700.0,0.0
17,K00018,46,신발,web,129900.0,3,389700.0,0.0
26,K00027,39,신발,app,129900.0,3,389700.0,0.0
34,K00035,34,상의,app,129900.0,3,389700.0,0.0
45,K00046,25,신발,app,129900.0,3,389700.0,0.0
98,K00099,42,하의,app,129900.0,3,389700.0,240917.0
213,K00214,33,하의,web,129900.0,3,389700.0,0.0
298,K00299,24,신발,app,129900.0,3,389700.0,0.0


In [10]:
partner.loc[547] #500만원 이상치 확인


order_id             K00548
customer_age             33
category                 신발
channel                 app
price               49900.0
quantity                  1
amount           50000000.0
return_amount           0.0
Name: 547, dtype: object

In [11]:
# quantity의 이상치 실제 값 확인
Q1 = partner["quantity"].quantile(0.25)
Q3 = partner["quantity"].quantile(0.75)
IQR = Q3 - Q1
lo = Q1 - 1.5 * IQR
up = Q3 + 1.5 * IQR

outliers_qty = partner[(partner["quantity"] < lo) | (partner["quantity"] > up)]
outliers_qty[["order_id", "category", "channel", "price", "quantity", "amount"]].sort_values("quantity", ascending=False)

,order_id,category,channel,price,quantity,amount
24,K00025,신발,app,29900.0,200,89700.0


In [12]:
# customer_age의 이상치 실제 값 확인
Q1 = partner["customer_age"].quantile(0.25)
Q3 = partner["customer_age"].quantile(0.75)
IQR = Q3 - Q1
lo = Q1 - 1.5 * IQR
up = Q3 + 1.5 * IQR

outliers_age = partner[(partner["customer_age"] < lo) | (partner["customer_age"] > up)]
outliers_age[["order_id", "customer_age", "category", "channel"]].sort_values("customer_age", ascending=False)

,order_id,customer_age,category,channel
75,K00076,999,상의,web
91,K00092,999,액세서리,web
1264,K01265,999,상의,web
703,K00704,60,신발,web
824,K00825,55,신발,web
757,K00758,54,액세서리,web
3,K00004,12,상의,web
1281,K01282,12,하의,app
636,K00637,12,상의,web
296,K00297,12,신발,app


In [13]:
# 어린 나이(0~12세) 그룹만 따로 채널 분포 확인
young = outliers_age[outliers_age["customer_age"] <= 12]
print(young["channel"].value_counts())
print("어린 나이 그룹 건수:", len(young))


channel
web    9
app    3
Name: count, dtype: int64
어린 나이 그룹 건수: 12


In [14]:
# 전체 데이터의 채널 비율과 비교
print("전체 채널 분포:")
print(partner["channel"].value_counts(normalize=True))

print("\n어린 나이 그룹(0~12세) 채널 분포:")
print(young["channel"].value_counts(normalize=True))

전체 채널 분포:
channel
app    0.606
web    0.394
Name: proportion, dtype: float64

어린 나이 그룹(0~12세) 채널 분포:
channel
web    0.75
app    0.25
Name: proportion, dtype: float64


In [15]:
# 반품 발생 여부와 비율 확인
print("반품 없음(0):", (partner["return_amount"] == 0).sum())
print("반품 있음(>0):", (partner["return_amount"] > 0).sum())
print("반품 비율:", (partner["return_amount"] > 0).mean() * 100, "%")

# 반품 있는 건들만 따로 보기
returned = partner[partner["return_amount"] > 0]
returned[["order_id", "amount", "return_amount", "category", "channel"]].sort_values("return_amount", ascending=False).head(10)

반품 없음(0): 1378
반품 있음(>0): 122
반품 비율: 8.133333333333333 %


,order_id,amount,return_amount,category,channel
1124,K01125,389700.0,322778.0,하의,app
1171,K01172,259800.0,249860.0,상의,app
526,K00527,259800.0,244322.0,상의,web
98,K00099,389700.0,240917.0,하의,app
804,K00805,259800.0,233379.0,하의,app
368,K00369,239700.0,231838.0,하의,app
217,K00218,259800.0,226327.0,신발,web
121,K00122,239700.0,220833.0,하의,app
939,K00940,259800.0,215962.0,상의,web
1039,K01040,239700.0,211426.0,하의,web


In [16]:
# 반품액이 결제액보다 큰 비정상 케이스 확인
weird = partner[partner["return_amount"] > partner["amount"]]
print("반품액 > 결제액인 이상 케이스:", len(weird), "건")
weird[["order_id", "amount", "return_amount"]]

반품액 > 결제액인 이상 케이스: 0 건


,order_id,amount,return_amount


In [17]:
# 시나리오 3 — 처리 코드 (예시 구현)
partner_clean = partner.copy()

# 1) customer_age 물리적 불가능 값 → NaN → 중앙값 대체
unrealistic = (partner_clean["customer_age"] < 1) | (partner_clean["customer_age"] > 110)
partner_clean.loc[unrealistic, "customer_age"] = np.nan
partner_clean["customer_age"] = partner_clean["customer_age"].fillna(
    partner_clean["customer_age"].median()
).astype(int)

# 2) quantity 이상치 → NaN → 중앙값 대체
mask_q, _ = detect_outliers_iqr(partner_clean["quantity"])
partner_clean.loc[mask_q, "quantity"] = np.nan
partner_clean["quantity"] = partner_clean["quantity"].fillna(
    partner_clean["quantity"].median()
).astype(int)

# 3) amount 이상치(50,000,000) → 유지 + 플래그
mask_a, _ = detect_outliers_iqr(partner_clean["amount"])
partner_clean["amount_outlier"] = mask_a.astype(int)

# 4) amount 결측 → 채널별 중앙값 대체 (MAR 가설)
partner_clean["amount"] = partner_clean["amount"].fillna(
    partner_clean.groupby("channel")["amount"].transform("median")
)

# 5) price 결측 → 카테고리별 중앙값 대체
partner_clean["price"] = partner_clean["price"].fillna(
    partner_clean.groupby("category")["price"].transform("median")
)

# 검증 출력
print("[처리 전 후 결측 비교]")
before = partner.isnull().sum()
after = partner_clean[partner.columns].isnull().sum()
print(pd.DataFrame({"before": before, "after": after}))

print("\n[처리 후 customer_age 범위]:",
      partner_clean["customer_age"].min(), "~", partner_clean["customer_age"].max())
print("[amount_outlier=1 건수]:", partner_clean["amount_outlier"].sum())

[처리 전 후 결측 비교]
               before  after
order_id            0      0
customer_age        0      0
category            0      0
channel             0      0
price               5      0
quantity            0      0
amount             51      0
return_amount       0      0

[처리 후 customer_age 범위]: 5 ~ 60
[amount_outlier=1 건수]: 145


In [18]:
# 1단계: 명백한 오류값 먼저 수정 (price × quantity로 재계산)
partner.loc[547, "amount"] = partner.loc[547, "price"] * partner.loc[547, "quantity"]

# 2단계: 그 다음에 결측치 채우기
partner["amount"] = partner.groupby("channel")["amount"].transform(
    lambda x: x.fillna(x.median())
)

In [19]:
partner["price"] = partner["price"].fillna(partner["price"].median())
#price 결측치 처리, 5건의 적은 결측치 & MCAR이므로 중앙값 대체


In [18]:
before = partner.isnull().sum()

In [19]:
partner["price"] = partner["price"].fillna(partner["price"].median())
partner["amount"] = partner.groupby("channel")["amount"].transform(
    lambda x: x.fillna(x.median())
)
# ... 그 외 처리 코드

In [20]:
print("[처리 전후 결측 비교]")
after = partner.isnull().sum()
comparison = pd.DataFrame({"before": before, "after": after})
print(comparison)

[처리 전후 결측 비교]
               before  after
order_id            0      0
customer_age        0      0
category            0      0
channel             0      0
price               5      0
quantity            0      0
amount             51      0
return_amount       0      0
